# Phase 0 — Setup (local PC)
**Project:** Where Does Label Noise Hurt Ensembles? A 0-1 Bias–Variance Decomposition of Bagging, Random Forests and Boosting under Symmetric, One-Sided and Boundary Label Noise

This notebook checks the local environment and prepares the project folders. Run every cell from top to bottom.

**Before you start** (one time only, in a terminal):
```
conda create -n labelnoise python=3.11 -y
conda run -n labelnoise pip install -r requirements.txt
conda run -n labelnoise python -m ipykernel install --user --name labelnoise --display-name "Python (labelnoise)"
```
Then in VS Code: **Select Kernel** (top right) → **Python (labelnoise)**.

## Step 1 — Check we are using the right Python
We use a separate conda environment with Python 3.11, because the system Python (3.14) is too new for some ML libraries.

In [4]:
import os, sys, platform
from pathlib import Path

print("Python:", platform.python_version())
print("Interpreter:", sys.executable)

assert sys.version_info[:2] == (3, 11), "Wrong kernel: choose 'Python (labelnoise)' in the top-right kernel picker."
print("OK: running in the labelnoise environment.")

Python: 3.11.17
Interpreter: /home/gravtion/miniconda3/envs/labelnoise/bin/python
OK: running in the labelnoise environment.


## Step 2 — Project folder
All data and results are saved in the project folder (the parent of `notebooks/`), so every later notebook finds them in the same place.

In [5]:
NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
print("Project folder:", BASE)

Project folder: /home/gravtion/VS Code/College/ML Research Paper


## Step 3 — Import everything and record the versions
The paper's *Methodology* section must report these versions so others can reproduce the work. They are also saved to `config.json` in Step 4.

In [6]:
import time, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn, xgboost, openml, scipy

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import scikit_posthocs as sp

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "openml": openml.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
}
for k, v in versions.items():
    print(f"{k:13s} {v}")

python        3.11.17
numpy         2.4.6
pandas        3.0.6
scikit-learn  1.9.1
xgboost       3.2.0
openml        0.15.1
scipy         1.17.1
matplotlib    3.11.2


## Step 4 — Global settings and folder layout
One fixed seed makes every run repeatable. **Rule for all later code:** every random step takes a seed derived from `SEED`; never call a random function without one.

```
ML Research Paper/
├── notebooks/         00_setup.ipynb, 02_data.ipynb, ...
├── data/              cleaned datasets (.npz)          → Phase 2
├── results/
│   ├── raw/           one CSV per dataset              → Phases 4–6
│   ├── figures/       plots for the paper              → Phase 7
│   └── tables/        tables for the paper             → Phase 7
└── config.json        seed + library versions
```

In [7]:
SEED = 42
np.random.seed(SEED)

DIRS = {
    "data":    BASE / "data",
    "raw":     BASE / "results" / "raw",
    "figures": BASE / "results" / "figures",
    "tables":  BASE / "results" / "tables",
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

config = {"SEED": SEED, "n_jobs": -1, "cpu_threads": os.cpu_count(), "versions": versions}
(BASE / "config.json").write_text(json.dumps(config, indent=2))

for name, path in DIRS.items():
    print(f"{name:8s} {path}")
print("Saved:", BASE / "config.json")

data     /home/gravtion/VS Code/College/ML Research Paper/data
raw      /home/gravtion/VS Code/College/ML Research Paper/results/raw
figures  /home/gravtion/VS Code/College/ML Research Paper/results/figures
tables   /home/gravtion/VS Code/College/ML Research Paper/results/tables
Saved: /home/gravtion/VS Code/College/ML Research Paper/config.json


## Step 5 — Check the CPU and time each model
`n_jobs=-1` lets Random Forest, Bagging and XGBoost use all CPU threads. Phase 4 uses these timings to estimate how long the full experiment will take.

In [8]:
from sklearn.datasets import make_classification

print("CPU threads available:", os.cpu_count())

X, y = make_classification(n_samples=2000, n_features=20, random_state=SEED)
models = {
    "Tree":     DecisionTreeClassifier(random_state=SEED),
    "Bagging":  BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, n_jobs=-1, random_state=SEED),
    "RF":       RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=SEED),
    "GB":       GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    "XGB":      XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, n_jobs=-1, random_state=SEED),
}
timings = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X, y)
    timings[name] = time.perf_counter() - start
    print(f"{name:9s} {timings[name]:.3f} s per fit (2,000 rows)")

CPU threads available: 12
Tree      0.028 s per fit (2,000 rows)
Bagging   2.337 s per fit (2,000 rows)
RF        0.152 s per fit (2,000 rows)
AdaBoost  0.461 s per fit (2,000 rows)
GB        0.747 s per fit (2,000 rows)
XGB       2.228 s per fit (2,000 rows)


## Step 6 — Save the timings and read them back
This confirms results are written to disk where later phases expect them.

In [9]:
test_path = DIRS["raw"] / "_setup_test.csv"
pd.DataFrame([{"model": k, "seconds_per_fit": round(v, 4)} for k, v in timings.items()]).to_csv(test_path, index=False)

df = pd.read_csv(test_path)
print("Saved and read back:", test_path)
df

Saved and read back: /home/gravtion/VS Code/College/ML Research Paper/results/raw/_setup_test.csv


,model,seconds_per_fit
0,Tree,0.0278
1,Bagging,2.3374
2,RF,0.1524
3,AdaBoost,0.4613
4,GB,0.7466
5,XGB,2.2282


## Phase 0 checklist
- [ ] Step 1 says *OK: running in the labelnoise environment*
- [ ] Versions printed (Step 3) and `config.json` created (Step 4)
- [ ] Timings shown and saved to `results/raw/_setup_test.csv` (Steps 5–6)

When all three are done, Phase 0 is complete. **Next: Phase 2 (datasets)**; Phase 1 (reading) can run in parallel.

**Before long runs later (Phases 5–6):** keep the laptop plugged in and turn off automatic sleep.